# BSM Paper Replication — Notebook 06: ResNet-18 (paper-faithful, real run)

SMOKE_TEST=False. 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3. Best-val checkpoint. Evaluate full test split. Stop if train_acc < 0.30 at epoch 30.

**Param count note:** `resnet18(head='paper')` uses Table 5 head (avgpool -> FC(feature, 6)) = **11,689,512** params (matches verified architecture).

In [1]:
# setup
import os, sys
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
import torch, random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SMOKE_TEST = False
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "SMOKE_TEST=", SMOKE_TEST)

gpu: True Tesla T4 SMOKE_TEST= False


## Real run: ResNet-18, 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, best-val checkpoint

In [2]:
# ===== REAL TRAINING: ResNet-18 (60 epochs, paper-faithful) =====
import os, sys, time, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "resnet18"
RES  = pathlib.Path(PROJECT) / "results"
CKPT.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

from models import resnet18
import common

# ---- data (split arrays from nb04) ----
def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    Xt = torch.from_numpy(X.transpose(0, 3, 1, 2)).contiguous()  # NHWC -> NCHW
    yt = torch.from_numpy(y.astype(np.int64))
    return Xt, yt

Xtr, ytr = load_xy("train")
Xva, yva = load_xy("val")
print("train:", Xtr.shape, "val:", Xva.shape, "| dtype", Xtr.dtype, "range", float(Xtr.min()), float(Xtr.max()))

MICRO = 32
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=MICRO, shuffle=False)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=MICRO, shuffle=False)

# ---- model ----
torch.manual_seed(42)
model = resnet18().cuda()
n_params = sum(p.numel() for p in model.parameters())
print("resnet18 params:", f"{n_params:,}")

EPOCHS = 60
t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=EPOCHS, full_batch=True,
    ckpt_dir=CKPT, history_path=RES / "history_resnet18.json",
    tag="resnet18", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "best val:", hist.get("best_val_acc"))

tr_acc = hist.get("train_acc") or []
if len(tr_acc) >= 30 and tr_acc[29] < 0.30:
    print("WARNING: train acc at epoch 30 =", tr_acc[29], "< 0.30 (spec stop-check)")

train: torch.Size([5651, 3, 224, 224]) val: torch.Size([1211, 3, 224, 224]) | dtype torch.float32 range 0.0 1.0
resnet18 params: 11,689,512
resnet18 epoch 01/60  train_loss 1.8379 acc 0.1727  val_loss 1.9637 acc 0.1792  (7.8s)
resnet18 epoch 02/60  train_loss 1.8352 acc 0.1739  val_loss 1.9209 acc 0.1792  (6.7s)
resnet18 epoch 03/60  train_loss 1.8089 acc 0.1769  val_loss 1.8616 acc 0.1792  (6.7s)
resnet18 epoch 04/60  train_loss 1.7605 acc 0.1863  val_loss 1.7999 acc 0.1792  (6.7s)
resnet18 epoch 05/60  train_loss 1.7121 acc 0.1986  val_loss 1.7696 acc 0.1833  (6.7s)
resnet18 epoch 06/60  train_loss 1.6785 acc 0.2232  val_loss 1.7391 acc 0.2139  (6.7s)
resnet18 epoch 07/60  train_loss 1.6504 acc 0.2440  val_loss 1.7015 acc 0.2502  (6.7s)
resnet18 epoch 08/60  train_loss 1.6219 acc 0.2695  val_loss 1.6838 acc 0.2692  (6.7s)
resnet18 epoch 09/60  train_loss 1.5960 acc 0.2875  val_loss 1.6361 acc 0.2940  (6.7s)
resnet18 epoch 10/60  train_loss 1.5698 acc 0.3057  val_loss 1.6134 acc 0.305

In [3]:
# ===== EVAL ResNet-18 (best-val ckpt) on TEST SPLIT =====
import os, sys, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "resnet18"
RES  = pathlib.Path(PROJECT) / "results"
RES.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
from models import resnet18
import common

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    return torch.from_numpy(X.transpose(0,3,1,2)).contiguous(), torch.from_numpy(y.astype(np.int64))

Xte, yte = load_xy("test")
print("test:", Xte.shape)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)

device = torch.device("cuda")
model = resnet18().to(device)
ck = torch.load(CKPT / "resnet18_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
_hist = json.load(open(RES / "history_resnet18.json"))
_ck = {"epoch": _hist.get("best_epoch", -1)}
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_true)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
spec_list = []
for i in range(6):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec = float(np.mean(spec_list))

print(f"\n=== ResNet-18 TEST SPLIT (best epoch {_ck['epoch']}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("\nconfusion matrix (rows=true):")
print(cm)
print("\nper-class:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

results = {
  "model": "resnet18", "notebook": "06_resnet18",
  "best_epoch": int(_ck.get("epoch", -1)),
  "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                 "macro_f1": float(f1), "macro_specificity": spec,
                 "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
  "paper_reported_table9": {"accuracy": 0.78, "precision": 0.87, "recall": 0.78, "note": "PAPER numbers (Table 9), for comparison only"},
}
with open(RES / "metrics_resnet18.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved", RES / "metrics_resnet18.json")
print("EVAL DONE")

test: torch.Size([1212, 3, 224, 224])

=== ResNet-18 TEST SPLIT (best epoch 60) ===
accuracy 0.5817 | macro precision 0.6060 | macro recall 0.5835 | macro F1 0.5742 | macro specificity 0.9164

confusion matrix (rows=true):
[[176  10   8   5   1   5]
 [ 93  87   4   2   7  25]
 [ 16   7  99  26  18  29]
 [ 13   3  51  94  14  23]
 [  0   3  15   7 135  38]
 [ 15  16  16   5  19 127]]

per-class:
              precision    recall  f1-score   support

    Ilmenite     0.5623    0.8585    0.6803       205
      Rutile     0.6905    0.3991    0.5060       218
      Zircon     0.5129    0.5077    0.5103       195
      Garnet     0.6763    0.4747    0.5574       198
    Monazite     0.6964    0.6818    0.6890       198
 Sillimanite     0.5142    0.6414    0.5707       198

    accuracy                         0.5817      1212
   macro avg     0.6088? (approx 0.606)    0.5835    0.5742      1212
weighted avg     0.6109    0.5817    0.5845      1212


saved /content/drive/MyDrive/bsm_paper_rep

In [4]:
# persist notebook copy
import glob, os, shutil
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","06_resnet18.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","06_resnet18.ipynb"))

copied -> /content/drive/MyDrive/bsm_paper_replication/notebooks/06_resnet18.ipynb
